# 02 — Auditoria dos dados (M2)

Pré-requisito: `python scripts/preprocess.py` (gera `data/processed/`) e, opcionalmente, `python scripts/audit_sample.py` + planilha `rotulos_manuais.csv` preenchida.

**Perguntas que este notebook responde**
1. O pré-processamento está íntegro (contagens, formas, máscaras binárias)?
2. Há imagens duplicadas ou quase duplicadas — em especial entre treino e teste (interno ou oficial)?
3. Com que frequência aparecem artefatos (vinheta, pelo, régua…), por classe e por split? Os detectores automáticos concordam com a rotulagem manual?
4. As augmentations produzem imagens plausíveis e mantêm a máscara alinhada?
5. Quais pesos de classe o treino deve usar?

Convenção: **medido** / **derivado** / **inferido**, como no notebook 01.

In [ ]:
import sys, json, time
from pathlib import Path
import numpy as np, pandas as pd, cv2, yaml
import matplotlib.pyplot as plt, seaborn as sns
from PIL import Image

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from skinlesion.artifacts import all_scores
from skinlesion.data import ClassificationDataset, SegmentationDataset, denormalize, load_yaml, mask_path

DATA = load_yaml("configs/data.yaml"); PRE = load_yaml("configs/preprocess.yaml"); TR = load_yaml("configs/transforms.yaml")
CLASSES = DATA["classes"]; SEED = DATA["seed"]; rng = np.random.default_rng(SEED)
FIG = ROOT / "reports/figures"; FIG.mkdir(parents=True, exist_ok=True); OUT = ROOT / "reports/m2_dados"; OUT.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid"); plt.rcParams["figure.dpi"] = 110
EVID = {}

man = pd.read_csv(ROOT / PRE["output"]["manifest"]); man["path"] = man.path.str.replace("\\", "/", regex=False)
gt = pd.read_csv(ROOT / DATA["task3_labels"]); gt.columns = [c.strip() for c in gt.columns]; gt["dx"] = gt[CLASSES].idxmax(axis=1)
gtt = pd.read_csv(ROOT / DATA["task3_test_labels"]); gtt.columns = [c.strip() for c in gtt.columns]; gtt["dx"] = gtt[CLASSES].idxmax(axis=1)
lab = pd.concat([gt[["image", "dx"]], gtt[["image", "dx"]]])
man = man.merge(lab, left_on="id", right_on="image", how="left").drop(columns="image")
print(man.groupby(["task", "source", "split"]).size())

## 1. Integridade do pré-processamento

Confere: todas as imagens brutas viraram processadas; formas corretas; máscaras continuam binárias; nenhum arquivo corrompido.

In [ ]:
esperado = {("cls", "train"): 10015, ("cls", "test_official"): 1512, ("seg", "task1"): 2594}
for (t, s), n in esperado.items():
    got = ((man.task == t) & (man.source == s) & man.status.isin(["ok", "skip"])).sum()
    print(f"{t}/{s}: {got} de {n} esperados  {'OK' if got == n else 'FALTANDO'}")
EVID["counts"] = {f"{t}/{s_}": int(n) for (t, s_), n in man.groupby(["task", "source"]).size().items()}

# amostra de 300 arquivos: forma e binariedade
amostra = man.sample(min(300, len(man)), random_state=SEED)
formas, nbin = {}, 0
for _, r in amostra.iterrows():
    im = cv2.imread(str(ROOT / r.path)); formas[r.task] = formas.get(r.task, set()) | {im.shape}
    if r.task == "seg":
        m = cv2.imread(str(ROOT / mask_path(r.path)), 0)
        nbin += set(np.unique(m).tolist()) <= {0, 255}
print("formas por task:", formas)
print("máscaras binárias na amostra:", nbin, "/", (amostra.task == "seg").sum())
EVID["shapes"] = {k: [list(x) for x in v] for k, v in formas.items()}

## 2. Duplicatas e quase-duplicatas (hash perceptual)

`pHash` resume a imagem em 64 bits; imagens idênticas ou quase (recompressão, leve redimensionamento) ficam a poucos bits de distância (Hamming). Procuramos pares com distância ≤ 4 em **todas** as imagens de classificação (treino interno + teste oficial) e na Task 1.

Por que importa: uma quase-duplicata entre treino e teste é vazamento que o `lesion_id` não captura (lesões diferentes, mesma foto? ou mesma lesão com ids diferentes).

In [ ]:
import imagehash
def phash(p): return imagehash.phash(Image.open(ROOT / p).convert("RGB"), hash_size=8)
t0 = time.time()
man["phash"] = [phash(p) for p in man.path]
print(f"{len(man)} hashes em {time.time()-t0:.0f}s")

def pares_proximos(df, max_dist=4):
    # bucket por 16 bits iniciais reduz comparações; depois Hamming completo dentro do bucket e vizinhos
    hs = np.array([h.hash.flatten() for h in df.phash], dtype=np.uint8)
    ints = np.packbits(hs, axis=1)
    pares = []
    for i in range(len(df)):
        d = np.unpackbits(ints[i] ^ ints[i+1:], axis=1).sum(1)
        for j in np.where(d <= max_dist)[0]:
            pares.append((df.index[i], df.index[i+1+j], int(d[j])))
    return pd.DataFrame(pares, columns=["a", "b", "dist"])

cls_all = man[man.task == "cls"].reset_index(drop=True)
dup = pares_proximos(cls_all)
dup["id_a"] = cls_all.loc[dup.a, "id"].values; dup["id_b"] = cls_all.loc[dup.b, "id"].values
dup["split_a"] = cls_all.loc[dup.a, "split"].values; dup["split_b"] = cls_all.loc[dup.b, "split"].values
dup["dx_a"] = cls_all.loc[dup.a, "dx"].values; dup["dx_b"] = cls_all.loc[dup.b, "dx"].values
print(f"pares cls com distância ≤4: {len(dup)} | exatos (0): {(dup.dist==0).sum()}")
cross = dup[dup.split_a != dup.split_b]
print(f"pares entre splits diferentes: {len(cross)}"); display(cross.head(15))
print("pares com rótulo diferente:", (dup.dx_a != dup.dx_b).sum())
EVID["dup_pairs_cls"] = int(len(dup)); EVID["dup_cross_split"] = int(len(cross)); EVID["dup_label_conflict"] = int((dup.dx_a != dup.dx_b).sum())

seg_all = man[man.task == "seg"].reset_index(drop=True)
dup_s = pares_proximos(seg_all)
print(f"pares seg com distância ≤4: {len(dup_s)}"); EVID["dup_pairs_seg"] = int(len(dup_s))
dup.to_csv(OUT / "duplicatas_cls.csv", index=False)

In [ ]:
# Visualiza os pares mais próximos entre splits (se houver) — para decidir o que fazer com eles
show = cross.sort_values("dist").head(6) if len(cross) else dup.sort_values("dist").head(6)
if len(show):
    fig, ax = plt.subplots(2, len(show), figsize=(2.6*len(show), 4.6), squeeze=False)
    for j, (_, r) in enumerate(show.iterrows()):
        for k, (i, s, d) in enumerate([(r.id_a, r.split_a, r.dx_a), (r.id_b, r.split_b, r.dx_b)]):
            ax[k, j].imshow(Image.open(ROOT / cls_all.set_index("id").loc[i, "path"])); ax[k, j].axis("off")
            ax[k, j].set_title(f"{i}\n{s} · {d}", fontsize=7)
        ax[0, j].set_title(f"dist={r.dist}\n" + ax[0, j].get_title(), fontsize=7)
    plt.tight_layout(); plt.savefig(FIG / "07_duplicatas.png"); plt.show()
else:
    print("nenhum par próximo")

## 3. Artefatos de aquisição: detectores automáticos em **todas** as imagens

Quatro scores por imagem (`src/skinlesion/artifacts.py`): brilho dos cantos (vinheta), fração de borda escura, fração de pixels em traços finos (pelo) e um proxy fraco de régua. Depois: taxa por classe e por split.

In [ ]:
t0 = time.time()
sc = pd.DataFrame([all_scores(cv2.imread(str(ROOT / p))) for p in man.path], index=man.index)
man = pd.concat([man.drop(columns=[c for c in sc.columns if c in man], errors="ignore"), sc], axis=1)
print(f"scores em {time.time()-t0:.0f}s"); display(man[sc.columns].describe().T.round(3))
man.drop(columns="phash").to_csv(OUT / "artefatos_scores.csv", index=False)

## 4. Rótulos manuais → limiares e qualidade dos detectores

Se `reports/m2_dados/auditoria/rotulos_manuais.csv` estiver preenchida (0/1 por artefato), escolhemos o limiar de cada score que maximiza o F1 contra o rótulo humano e reportamos precisão/recall. Sem a planilha, usamos limiares padrão e marcamos como **não validados**.

In [ ]:
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score
rot_path = ROOT / "reports/m2_dados/auditoria/rotulos_manuais.csv"
DETECTORES = {"vinheta": ("vignette", "menor"), "pelo": ("hair", "maior"), "regua": ("ruler", "maior")}
LIMIARES = {"vignette": 40.0, "hair": 0.012, "ruler": 200.0}   # padrão, revisado abaixo se houver rótulos
validado = {}
if rot_path.exists():
    rot = pd.read_csv(rot_path)
    preenchida = rot[list(DETECTORES)].notna().all(axis=1)
    rot = rot[preenchida]
    print(f"rótulos manuais: {len(rot)} imagens preenchidas")
    if len(rot) >= 30:
        r = rot.merge(man[["id", "task"] + list(sc.columns)], on=["id", "task"])
        for art, (score, sentido) in DETECTORES.items():
            y = r[art].astype(int).values; s = r[score].values if sentido == "maior" else -r[score].values
            if y.sum() == 0 or y.sum() == len(y): print(f"{art}: sem variação nos rótulos, pulando"); continue
            auc = roc_auc_score(y, s)
            best = max(((precision_recall_fscore_support(y, (s >= t).astype(int), average="binary", zero_division=0)[:3], t) for t in np.unique(s)), key=lambda z: z[0][2])
            (p, rc, f1), t = best; thr = t if sentido == "maior" else -t
            LIMIARES[score] = float(thr); validado[art] = dict(auc=round(auc, 3), precisao=round(p, 3), recall=round(rc, 3), f1=round(f1, 3), limiar=float(thr), prevalencia_manual=round(y.mean(), 3), n=int(len(y)))
            print(f"{art:8s} AUC={auc:.2f}  limiar={thr:.3f}  P={p:.2f} R={rc:.2f} F1={f1:.2f}  (prevalência manual {y.mean():.2f}, n={len(y)})")
        print("\nartefatos sem detector automático (só manual):", {c: round(rot[c].astype(float).mean(), 3) for c in ["bolha", "caneta"] if c in rot})
    else:
        print("menos de 30 rotuladas — limiares padrão, não validados")
else:
    print("planilha não encontrada — limiares padrão, não validados")
EVID["detectores_validados"] = validado; EVID["limiares"] = LIMIARES
yaml.safe_dump({"version": "v1", "limiares": LIMIARES, "validado_com_rotulos_manuais": bool(validado), "metricas": validado}, open(ROOT / "configs/artifacts.yaml", "w"), sort_keys=False)
print("salvo configs/artifacts.yaml")

In [ ]:
man["tem_vinheta"] = man.vignette < LIMIARES["vignette"]
man["tem_pelo"] = man.hair > LIMIARES["hair"]
man["tem_regua"] = man.ruler > LIMIARES["ruler"]
flags = ["tem_vinheta", "tem_pelo", "tem_regua"]
cls = man[(man.task == "cls") & (man.source == "train")]
por_classe = cls.groupby("dx")[flags].mean().reindex(CLASSES).round(3)
por_split = man[man.task == "cls"].groupby("split")[flags].mean().round(3)
print("taxa de artefato por classe (treino):"); display(por_classe)
print("taxa por split (inclui teste oficial):"); display(por_split)
EVID["artefato_por_classe"] = por_classe.to_dict(); EVID["artefato_por_split"] = por_split.to_dict()

fig, ax = plt.subplots(1, 3, figsize=(13, 3.4))
for a, f in zip(ax, flags):
    sns.barplot(x=por_classe.index, y=por_classe[f].values * 100, ax=a, color="#3b82f6"); a.set(title=f.replace("tem_", "") + " (% por classe)", xlabel="", ylabel="%")
plt.tight_layout(); plt.savefig(FIG / "08_artefatos_por_classe.png"); plt.show()

# teste qui-quadrado: o artefato é independente da classe?
from scipy.stats import chi2_contingency
for f in flags:
    chi2, p, _, _ = chi2_contingency(pd.crosstab(cls.dx, cls[f]))
    print(f"{f}: chi2={chi2:.1f}  p={p:.2e}  → {'associado à classe' if p < 0.001 else 'sem evidência de associação'}")
    EVID[f"chi2_{f}"] = dict(chi2=round(float(chi2), 1), p=float(p))

## 5. Augmentations: o que a rede vai ver

Mesma imagem, 6 sorteios da pipeline de treino; para segmentação, verifica que a máscara acompanha a geometria. Se algo parecer irreal (cor destruída, lesão cortada fora), o lugar de ajustar é `configs/transforms.yaml`, não o código.

In [ ]:
dsc = ClassificationDataset("train"); dss = SegmentationDataset("train")
i_c = int(rng.integers(len(dsc))); i_s = int(rng.integers(len(dss)))
fig, ax = plt.subplots(3, 7, figsize=(15, 6.6))
orig = cv2.cvtColor(cv2.imread(str(ROOT / dsc.m.iloc[i_c].path)), cv2.COLOR_BGR2RGB)
ax[0, 0].imshow(orig); ax[0, 0].set_title(f"original {dsc.m.iloc[i_c].dx}", fontsize=8)
for j in range(1, 7):
    x, y, _ = dsc[i_c]; ax[0, j].imshow(denormalize(x)); ax[0, j].set_title("aug treino", fontsize=8)
orig = cv2.cvtColor(cv2.imread(str(ROOT / dss.m.iloc[i_s].path)), cv2.COLOR_BGR2RGB)
m0 = cv2.imread(str(ROOT / mask_path(dss.m.iloc[i_s].path)), 0)
ax[1, 0].imshow(orig); ax[2, 0].imshow(orig); ax[2, 0].imshow(m0 > 127, alpha=.4, cmap="Reds"); ax[1, 0].set_title("original seg", fontsize=8)
for j in range(1, 7):
    x, m, _ = dss[i_s]; im = denormalize(x); ax[1, j].imshow(im); ax[2, j].imshow(im); ax[2, j].imshow(m[0].numpy(), alpha=.4, cmap="Reds")
for a in ax.ravel(): a.axis("off")
plt.suptitle("linha 1: classificação · linhas 2-3: segmentação (imagem e máscara sobreposta)"); plt.tight_layout(); plt.savefig(FIG / "09_augmentations.png"); plt.show()
print("transforms de treino (cls):", [s["name"] for s in TR["classification"]["train"]])
print("transforms de treino (seg):", [s["name"] for s in TR["segmentation"]["train"]])

## 6. Pesos de classe e checagem final do split

Pesos inversamente proporcionais à frequência (normalizados para média 1), salvos para o experimento comparativo da M3/M4. Também confirma nos dados **processados** que nenhuma lesão está em dois splits.

In [ ]:
cnt = dsc.class_counts(); w = (cnt.sum() / (len(CLASSES) * cnt)); w = (w / w.mean()).round(3)
pesos = {"version": "v1", "metodo": "inverso da frequencia, media 1", "contagem_treino": cnt.to_dict(), "pesos": w.to_dict()}
json.dump(pesos, open(ROOT / "data/splits/class_weights_v1.json", "w"), indent=2); print(pd.DataFrame({"n_treino": cnt, "peso": w}))

grp = pd.read_csv(ROOT / DATA["task3_groupings"])
chk = man[(man.task == "cls") & (man.source == "train")].merge(grp[["image", "lesion_id"]], left_on="id", right_on="image")
vaz = chk.groupby("lesion_id").split.nunique()
print("\nlesões em mais de um split (dados processados):", int((vaz > 1).sum())); assert (vaz == 1).all()
EVID["class_weights"] = w.to_dict(); EVID["leak_check_processed"] = int((vaz > 1).sum())

## 7. Evidências

In [ ]:
json.dump(EVID, open(OUT / "evidencias_m2.json", "w"), indent=2, ensure_ascii=False, default=str)
print(json.dumps(EVID, indent=2, ensure_ascii=False, default=str))